In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

###############################################################################
# ABJM Charges Playground (Cartan-basis inputs -> derive everything)
# ============================================================================
#
# GOAL OF THIS SCRIPT
# -------------------
# We have a list of ABJM “letters” (fields like Y^1, psi_{1+}, D_+, ...).
#
# For each letter we want a table of charges.
#
# We do NOT want to type every single charge by hand.
# Instead, we type only a small set of fundamental charges per letter,
# and compute everything else automatically from those.
#
#
# WHAT WE TYPE IN BY HAND FOR EACH LETTER
# ---------------------------------------
#   H1c, H2c, H3c, H1a, H2a, b
#
# These are the only independent inputs in this version of the script.
#
# IMPORTANT CHANGE COMPARED TO THE OLD VERSION
# --------------------------------------------
# In the old version we also typed:
#
#   J_gamma_c
#
# as an independent input.
#
# But that is now removed completely, because we now know that
#
#   J_gamma_c = b
#
# so it was redundant information.
#
# Therefore:
#   - we no longer store J_gamma_c anywhere,
#   - we no longer ask for it in the input rows,
#   - and whenever we need the “sum of the J_a’s”, we simply use b.
#
#
# WHAT WE COMPUTE AUTOMATICALLY
# -----------------------------
# 1) J1, J2, J3, J4
#    These are the 4 SO(8)/S^7 angular momenta.
#
#    They are determined by the linear system:
#
#      H1c = 1/2 ( J1 - J2 - J3 + J4 )
#      H2c = 1/2 (-J1 + J2 - J3 + J4 )
#      H3c = 1/2 (-J1 - J2 + J3 + J4 )
#      b   =       J1 + J2 + J3 + J4
#
#    Since b plays the role previously assigned to J_gamma_c,
#    we solve for J1..J4 using (H1c, H2c, H3c, b).
#
# 2) p1, q, p2
#    The SU(4)_R weight labels used in Klose’s conventions.
#    These are computed directly from H^c.
#
# 3) SU2_G', SU2_G, U1_extra
#    Also computed directly from H^c.
#
# 4) Delta and s
#    These are computed from H^a.
#
# 5) E and E_new
#    Two energy-like combinations built from the same Cartans.
#
#
# OUTPUT
# ------
# We can print the final table as:
#   - a plain text table
#   - a pandas DataFrame
#   - or both
#
# All numbers are exact Fractions, never floats.
#
#
# IMPORTANT SPECIAL RULE FOR DERIVATIVES
# --------------------------------------
# Derivative letters D_0, D_+, D_- are special in your convention:
#
#   - for them, the H^a entries are interpreted as shifts,
#   - therefore the “new energy” E_new should NOT include the extra +1 term.
#
# We implement exactly:
#
#   E_new = (H2a - H1c - H2c - H3c + shift)/2
#
# with:
#   shift = 1  for NON-derivative letters
#   shift = 0  for derivative letters D_*
###############################################################################

from __future__ import annotations

###############################################################################
# Standard library imports
###############################################################################
import argparse
import sys
from fractions import Fraction
from typing import Any, Dict, List, Optional, Sequence

###############################################################################
# Optional imports
#
# If pandas exists, we can return a DataFrame.
# If IPython display exists, notebooks can show the DataFrame nicely.
#
# If these packages are missing, the script still works fine and simply prints.
###############################################################################
try:
    import pandas as pd
except ImportError:
    pd = None  # type: ignore

try:
    from IPython.display import display as ipy_display  # type: ignore
except Exception:
    ipy_display = None  # type: ignore

###############################################################################
# Convenience alias:
#   Fr(...) creates exact rational numbers.
#
# Example:
#   Fr(1,2) is exactly 1/2
###############################################################################
Fr = Fraction

###############################################################################
# Default output mode:
#   "print" -> plain text table
#   "df"    -> pandas DataFrame (if pandas exists)
#   "both"  -> print + DataFrame
###############################################################################
DEFAULT_OUTPUT_MODE = "df"

###############################################################################
# Default columns to show
#
# Note:
#   J_gamma_c has been REMOVED completely.
#   If at some later stage you want to display “the sum of the J_a’s”,
#   that is now simply the column b.
###############################################################################
DEFAULT_COLS_CSV = (
    "Field,"
    "H1c,H2c,H3c,H1a,H2a,"
    "p1,q,p2,"
    "J1,J2,J3,J4,"
    "SU2_G',SU2_G,U1_extra,"
    "Delta,s,E,E_new,b"
)

###############################################################################
# BASE DATA TABLE
# -----------------------------------------------------------------------------
# This is the ONLY place where letter-specific data is entered by hand.
#
# Each dictionary is one ABJM letter.
#
# Hand-entered fields:
#   Field          : letter name
#   H1c, H2c, H3c  : SU(4)/SO(6)-side Cartans
#   H1a, H2a       : SO(2,3)/Sp(4)-side Cartans
#   b              : baryonic / site-parity-like charge
#
# Everything else is derived later.
#
# IMPORTANT:
#   We do NOT store J_gamma_c anymore, because it equals b.
###############################################################################
ROWS_BASE: List[Dict[str, Any]] = [
    # --- Scalars Y^A (odd-site matter, b=+1) ---
    dict(Field="Y^1", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(1,2),  H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="Y^2", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(1,2),  H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="Y^3", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(1,2),  H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="Y^4", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(1,2),  H2a=Fr(1,2), b=Fr( 1)),

    # --- Fermions psi_{A±} (odd-site matter, b=+1) ---
    dict(Field="psi_{1+}", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(3,2), H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="psi_{2+}", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(3,2), H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="psi_{3+}", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(3,2), H2a=Fr(1,2), b=Fr( 1)),
    dict(Field="psi_{4+}", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(3,2), H2a=Fr(1,2), b=Fr( 1)),

    dict(Field="psi_{1-}", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(3,2), b=Fr( 1)),
    dict(Field="psi_{2-}", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(3,2), b=Fr( 1)),
    dict(Field="psi_{3-}", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(3,2), b=Fr( 1)),
    dict(Field="psi_{4-}", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(3,2), b=Fr( 1)),

    # --- Derivatives (special shift convention, b=0) ---
    dict(Field="D_0", H1c=Fr(0), H2c=Fr(0), H3c=Fr(0), H1a=Fr(1), H2a=Fr(1), b=Fr(0)),
    dict(Field="D_+", H1c=Fr(0), H2c=Fr(0), H3c=Fr(0), H1a=Fr(2), H2a=Fr(0), b=Fr(0)),
    dict(Field="D_-", H1c=Fr(0), H2c=Fr(0), H3c=Fr(0), H1a=Fr(0), H2a=Fr(2), b=Fr(0)),

    # --- Conjugate scalars (even-site matter, b=-1) ---
    dict(Field="Y^dagger_1", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="Y^dagger_2", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="Y^dagger_3", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="Y^dagger_4", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(1,2), b=-Fr(1)),

    # --- Conjugate fermions (even-site matter, b=-1) ---
    dict(Field="psi^{dagger 1+}", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(3,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="psi^{dagger 2+}", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(3,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="psi^{dagger 3+}", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(3,2), H2a=Fr(1,2), b=-Fr(1)),
    dict(Field="psi^{dagger 4+}", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(3,2), H2a=Fr(1,2), b=-Fr(1)),

    dict(Field="psi^{dagger 1-}", H1c=Fr(1,2),  H2c=Fr(1,2),  H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(3,2), b=-Fr(1)),
    dict(Field="psi^{dagger 2-}", H1c=Fr(1,2),  H2c=-Fr(1,2), H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(3,2), b=-Fr(1)),
    dict(Field="psi^{dagger 3-}", H1c=-Fr(1,2), H2c=Fr(1,2),  H3c=-Fr(1,2), H1a=Fr(1,2), H2a=Fr(3,2), b=-Fr(1)),
    dict(Field="psi^{dagger 4-}", H1c=-Fr(1,2), H2c=-Fr(1,2), H3c=Fr(1,2),  H1a=Fr(1,2), H2a=Fr(3,2), b=-Fr(1)),
]

###############################################################################
# Helper: Fraction -> nice string
#
# We print integers as "2" instead of "2/1".
# Non-integers are printed as "a/b".
###############################################################################
def frac_to_str(x: Fraction) -> str:
    if x.denominator == 1:
        return str(x.numerator)
    return f"{x.numerator}/{x.denominator}"

###############################################################################
# Compute J1..J4 from (H1c, H2c, H3c, b)
# -----------------------------------------------------------------------------
# We solve the linear system
#
#   H1c = 1/2 ( J1 - J2 - J3 + J4 )
#   H2c = 1/2 (-J1 + J2 - J3 + J4 )
#   H3c = 1/2 (-J1 - J2 + J3 + J4 )
#   b   =       J1 + J2 + J3 + J4
#
# Why do we need b here?
# ----------------------
# The first three equations only fix J1..J4 up to adding the SAME constant
# to all four J's. That common shift leaves H1c, H2c, H3c unchanged.
#
# The final equation involving b fixes that remaining freedom.
#
# In the old code this was implemented as:
#   "sum-zero base solution" + "uniform shift"
#
# That was mathematically correct, but harder to read.
#
# Here we instead write the final direct formulas explicitly.
###############################################################################
def compute_Js_from_cartans(
    H1c: Fraction,
    H2c: Fraction,
    H3c: Fraction,
    b: Fraction,
) -> Dict[str, Fraction]:
    return {
        "J1": (2*H1c - 2*H2c - 2*H3c + b) / 4,
        "J2": (-2*H1c + 2*H2c - 2*H3c + b) / 4,
        "J3": (-2*H1c - 2*H2c + 2*H3c + b) / 4,
        "J4": (2*H1c + 2*H2c + 2*H3c + b) / 4,
    }

###############################################################################
# Add all derived columns to one row
# -----------------------------------------------------------------------------
# Input row contains only the hand-entered base data.
# This function mutates the row in-place by adding all computed quantities.
###############################################################################
def add_derived_columns(r: Dict[str, Any]) -> None:
    # Read the fundamental inputs from the row
    H1c, H2c, H3c = r["H1c"], r["H2c"], r["H3c"]
    H1a, H2a = r["H1a"], r["H2a"]
    b = r["b"]

    # (1) Compute J1..J4
    #     These now come from (H1c, H2c, H3c, b)
    r.update(compute_Js_from_cartans(H1c, H2c, H3c, b))

    # (2) Compute Klose-style SU(4)_R labels from H^c
    r["p1"] = H2c + H3c
    r["q"]  = H1c - H2c
    r["p2"] = H2c - H3c

    # (3) Compute the alternative SU(2)xSU(2)xU(1)-style combinations
    r["SU2_G'"]   = (H1c + H2c) / 2
    r["SU2_G"]    = (H1c - H2c) / 2
    r["U1_extra"] = H3c

    # (4) Compute conformal dimension and spin from H^a
    r["Delta"] = (H1a + H2a) / 2
    r["s"]     = (H1a - H2a) / 2

    # (5) Old energy:
    #       E = Delta - J
    #   with J = (H1c + H2c)/2 in your convention
    r["E"] = (H1a + H2a) / 2 - (H1c + H2c) / 2

    # (6) New energy:
    #       E_new = (H2a - H1c - H2c - H3c + shift)/2
    #
    #   For ordinary letters: shift = 1
    #   For derivative letters: shift = 0
    is_derivative = str(r["Field"]).startswith("D_")
    shift = 0 if is_derivative else 1
    r["E_new"] = (H2a - H1c - H2c - H3c + shift) / 2

###############################################################################
# Build the full table
# -----------------------------------------------------------------------------
# We copy each base row and then append all derived quantities.
# The base table ROWS_BASE itself is left untouched.
###############################################################################
def build_table() -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for row in ROWS_BASE:
        r = dict(row)           # shallow copy of the input row
        add_derived_columns(r)  # derive everything else
        out.append(r)
    return out

###############################################################################
# Print a plain ASCII table
# -----------------------------------------------------------------------------
# This is useful when:
#   - pandas is unavailable
#   - or you want a nice aligned terminal-style printout
###############################################################################
def print_table_plain(rows: List[Dict[str, Any]], cols: List[str]) -> None:
    # First determine the display width of each column
    widths: Dict[str, int] = {}
    for c in cols:
        maxlen = len(c)
        for r in rows:
            v = r.get(c, "")
            v_str = frac_to_str(v) if isinstance(v, Fraction) else str(v)
            maxlen = max(maxlen, len(v_str))
        widths[c] = maxlen

    # Header row
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    print(header)
    print("-" * len(header))

    # Data rows
    for r in rows:
        parts = []
        for c in cols:
            v = r.get(c, "")
            v_str = frac_to_str(v) if isinstance(v, Fraction) else str(v)
            parts.append(v_str.ljust(widths[c]))
        print(" | ".join(parts))

###############################################################################
# Build a pandas DataFrame
# -----------------------------------------------------------------------------
# Fractions are converted to strings for prettier display.
###############################################################################
def make_dataframe(rows: List[Dict[str, Any]], cols: List[str]):
    if pd is None:
        return None

    df = pd.DataFrame(rows)

    # Convert Fraction objects to strings so the DataFrame display is readable
    for c in cols:
        if c in df.columns:
            df[c] = df[c].apply(lambda x: frac_to_str(x) if isinstance(x, Fraction) else x)

    return df[cols]

###############################################################################
# Print a DataFrame as plain text
###############################################################################
def print_dataframe_as_text(df) -> None:
    print(df.to_string(index=False))

###############################################################################
# Main runner
# -----------------------------------------------------------------------------
# This function:
#   - chooses which columns to show,
#   - builds the full table,
#   - returns or prints the result depending on output mode.
###############################################################################
def run(cols: Optional[Sequence[str]] = None, output: str = DEFAULT_OUTPUT_MODE):
    # Decide which columns should be shown
    if cols is None:
        cols_list = [c.strip() for c in DEFAULT_COLS_CSV.split(",") if c.strip()]
    else:
        cols_list = list(cols)

    # Build the fully derived table
    rows = build_table()

    # This line is kept because you used this table-order idea later
    # in your notebook / Sage workflow.
    TABLE_ORDER = [str(r["Field"]) for r in rows]

    # Validate output mode
    output = output.strip().lower()
    if output not in ("print", "df", "both"):
        raise ValueError('output must be one of: "print", "df", "both"')

    # Mode 1: plain text only
    if output == "print":
        print_table_plain(rows, cols_list)
        return None

    # Mode 2 or 3: build DataFrame if possible
    df = make_dataframe(rows, cols_list)

    # If pandas is unavailable, fall back to plain printing
    if df is None:
        print_table_plain(rows, cols_list)
        return None

    # If requested, also print the DataFrame as plain text
    if output == "both":
        print_dataframe_as_text(df)

    # If in a notebook, display nicely
    if ipy_display is not None:
        ipy_display(df)

    return df

###############################################################################
# Jupyter-safe CLI argument cleanup
# -----------------------------------------------------------------------------
# Some notebook kernels launch Python with arguments like:
#
#   -f something.json
#
# argparse would otherwise complain about these.
# So we remove them before parsing our own arguments.
###############################################################################
def strip_ipykernel_argv(argv: List[str]) -> List[str]:
    cleaned: List[str] = []
    i = 0
    while i < len(argv):
        if argv[i] == "-f" and i + 1 < len(argv):
            i += 2
            continue
        cleaned.append(argv[i])
        i += 1
    return cleaned

###############################################################################
# CLI entry point
###############################################################################
def main() -> None:
    ap = argparse.ArgumentParser(
        prog="abjm_charges_cartan_basis.py",
        description=(
            "ABJM charge table: input Cartans (H1c,H2c,H3c,H1a,H2a,b), "
            "derive the rest exactly using Fractions."
        ),
    )

    # Choose and/or reorder visible columns
    ap.add_argument("--cols", default=None, help="Comma-separated columns to show.")

    # Select output style
    ap.add_argument(
        "--output",
        default=DEFAULT_OUTPUT_MODE,
        choices=["print", "df", "both"],
        help="Output mode."
    )

    # Parse arguments safely in notebook or normal Python
    argv = strip_ipykernel_argv(sys.argv[1:])
    args = ap.parse_args(argv)

    # Turn --cols into a Python list if supplied
    cols = [c.strip() for c in args.cols.split(",") if c.strip()] if args.cols else None

    # Run the table builder
    _ = run(cols=cols, output=args.output)

###############################################################################
# Standard Python entry point
###############################################################################
#if __name__ == "__main__":
#    main()

In [7]:
###############################################################################
# ABJM Sector Finder (SageMath) — FULL IMPLEMENTATION
# ===================================================
# Requires build_table() from the block above.
#
# IMPORTANT UPDATE
# ----------------
# In the old version of this workflow, some bases and conversion formulas used
# a quantity called J_gamma_c.
#
# That quantity has now been removed completely, because we now know:
#
#   J_gamma_c = b
#
# Therefore, throughout this entire block:
#
#   - J_gamma_c no longer appears anywhere,
#   - any old basis that used J_gamma_c now uses b instead,
#   - and the identity "sum of J_a = b" is used directly.
#
# Also:
#   - there is NO "include_b" switch anymore;
#   - b appears only if you explicitly include it in the basis itself.
#
# Finally:
#   - the coefficient of Delta is fixed to 1 by construction, because we search
#     sectors using
#
#         P(letter) = Delta(letter) - n · C(letter) >= 0
#
#     where n only contains the coefficients of the chosen basis charges.
#
# NEW IN THIS VERSION
# -------------------
# 1) Orbit-invariant n-based canonical representatives:
#
#      canonical_rep_mode = "letters"
#          -> uses the old letter-content preference score
#
#      canonical_rep_mode = "nvec"
#          -> chooses the representative whose OWN face admits the best
#             n-vector according to canonical_nvec_score(...)
#
#    This is orbit-invariant because the score is computed from the target
#    face itself, not from an arbitrary transformed point from some other face.
#
# 2) Final sorting of the already-unique sectors can be chosen independently:
#
#      final_sector_sort_mode = "letters" | "nvec"
#
# 3) The printed example point can be chosen independently:
#
#      example_point_mode = "pretty" | "nvec" | "raw"
#
# 4) Exact preferred values can be imposed as soft preferences for ANY n_i.
#
# 5) Sum-objectives can be used as soft preferences:
#      - maximize/minimize total sum over chosen coordinates
#      - include/exclude any subset (for example exclude n_s)
###############################################################################

from sage.all import *
import re
from itertools import permutations, combinations, product

QQ = RationalField()

# ---------------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------------
SETTINGS = {
    # Core
    "run_basis": "J5",              # key in BASES
    "force_zero": [],               # list of coefficient names to force to 0 (must be in RUN_COLS)

    # Vacuum / alternation filters
    "vacuum_mode": "none",          # "none" | "Y1_Yd4" | "Y1_psid1+"
    "require_alternating_feasible": True,

    # Quotients
    "quotient_pm_flip": True,
    "quotient_dagger_swap": True,
    "quotient_index_perm_S4": True,

    # Representative choice for quotienting
    # "letters" -> old letter-based representative
    # "nvec"    -> orbit-invariant face-based n-vector representative
    "canonical_rep_mode": "letters",

    # If canonical_rep_mode="letters", use the hand-made preference score
    "use_preferences_for_canonical_rep": True,

    # Final sorting of already-unique sectors
    # "letters" | "nvec"
    "final_sector_sort_mode": "letters",

    # Which example point to print?
    # "pretty" | "nvec" | "raw"
    "example_point_mode": "pretty",

    # Two-master system + identity policy
    "master_basis": "spin",          # "spin" or "J"
    "identities_mode": "use_for_conversion",  # "check_only" | "use_for_conversion"
    "run_identity_checks": True,

    # Evaluate mode
    "evaluate_mode": False,
    "evaluate_n": None,

    # ------------------------------------------------------------------
    # N-VECTOR PREFERENCES (used for canonical_rep_mode="nvec",
    # final_sector_sort_mode="nvec", and example_point_mode="nvec")
    #
    # All are SOFT preferences, not hard constraints.
    #
    # You can target ANY coordinates present in RUN_COLS, e.g.
    #   {"J4": QQ(1)/2, "s": QQ(0)}
    # or in a pq basis:
    #   {"p1": QQ(1)/2, "b": QQ(0)}
    # ------------------------------------------------------------------
    "nvec_prefer_exact_values": {},      # dict: coord -> exact preferred value
    "nvec_prefer_nonnegative": [],       # list of coords preferred >= 0
    "nvec_prefer_nonpositive": [],       # list of coords preferred <= 0
    "nvec_prefer_zero": [],              # list of coords preferred = 0

    "nvec_penalty_exact": 10000,
    "nvec_penalty_sign": 1000,
    "nvec_penalty_zero": 500,

    # Sum objective on chosen coordinates.
    #
    # "none"      -> ignore sums
    # "maximize"  -> prefer larger sum over chosen coords
    # "minimize"  -> prefer smaller sum over chosen coords
    #
    # The coordinates included are set by nvec_sum_coords.
    # If empty, all RUN_COLS are used.
    #
    # Examples:
    #   maximize all coords:
    #       "nvec_sum_mode": "maximize", "nvec_sum_coords": []
    #
    #   maximize all except s:
    #       "nvec_sum_mode": "maximize", "nvec_sum_coords": ["J1","J2","J3","J4"]
    #
    #   minimize only J1+J2+J3+J4:
    #       "nvec_sum_mode": "minimize", "nvec_sum_coords": ["J1","J2","J3","J4"]
    "nvec_sum_mode": "none",             # "none" | "maximize" | "minimize"
    "nvec_sum_coords": [],               # empty -> all RUN_COLS

    # ------------------------------------------------------------------
    # Pretty face-point search
    # ------------------------------------------------------------------
    "use_nice_example_interiors": False,

    # Search points of the form n = m/D with ONE common denominator D
    "nice_common_denominators": [1, 2, 3, 4, 6, 8, 12, 16],
    "nice_max_integer": 3,

    # Stop after this many candidate points per face (None = no cap)
    "nice_max_candidates_per_face": 50000,

    # Pretty-point preferences (same structure as nvec preferences)
    "nice_prefer_exact_values": {},
    "nice_prefer_nonnegative": [],
    "nice_prefer_nonpositive": [],
    "nice_prefer_zero": [],

    "nice_penalty_exact": 10000,
    "nice_penalty_sign": 1000,
    "nice_penalty_zero": 500,

    "nice_sum_mode": "none",             # "none" | "maximize" | "minimize"
    "nice_sum_coords": [],               # empty -> all RUN_COLS

    # Printing
    "max_sectors_to_print": None,
    "max_strict_ineq_to_print": 12,
    "print_charge_cols_in_diagnostics": True,
    "extra_cols_in_diagnostics": [],
    "print_n_in_other_bases": False,
    "print_raw_representative_if_different": False,

    # How to print n-vectors
    # "fraction"      -> only (a/b, c/d, ...)
    # "common_den"    -> only (1/D)*(...)
    # "both"          -> show both
    "n_vector_print_mode": "fraction",
}

# ---------------------------------------------------------------------------
# BASES (can be ANY length)
# ---------------------------------------------------------------------------
BASES = {
    # J-type
    "J4":    ["J1","J2","J3","J4"],
    "J5":    ["J1","J2","J3","J4","s"],

    # Spin-type
    "spin3": ["H1c","H2c","H3c"],
    "spin4": ["H1c","H2c","H3c","s"],
    "spin5": ["H1c","H2c","H3c","s","b"],

    # pq-type
    "pq3":   ["p1","q","p2"],
    "pq4":   ["p1","q","p2","s"],
    "pq5":   ["p1","q","p2","s","b"],
}

# ---------------------------------------------------------------------------
# Universal coordinate order
# ---------------------------------------------------------------------------
UNIVERSAL_ORDER = [
    "H1c","H2c","H3c","b",
    "J1","J2","J3","J4",
    "p1","q","p2",
    "s","Delta"
]
mU = len(UNIVERSAL_ORDER)

def basis_cols(bkey):
    if bkey not in BASES:
        raise ValueError(f"Unknown basis '{bkey}'. Known: {list(BASES.keys())}")
    return list(BASES[bkey])

# ---------------------------------------------------------------------------
# Load letters from build_table() (Fractions -> QQ)
# ---------------------------------------------------------------------------
def _toQQ(x):
    try:
        from fractions import Fraction
        if isinstance(x, Fraction):
            return QQ(x.numerator) / QQ(x.denominator)
    except Exception:
        pass
    return QQ(x)

def load_letters_from_table():
    rows = build_table()
    letters = {}
    for r in rows:
        nm = str(r["Field"])
        letters[nm] = {k: _toQQ(v) for k, v in r.items() if k != "Field"}
        if "Delta" not in letters[nm]:
            raise ValueError(f"Row {nm} missing Delta")
    return letters

LETTERS = load_letters_from_table()
ALL_NAMES = sorted(LETTERS.keys())

# ---------------------------------------------------------------------------
# Site classification
# ---------------------------------------------------------------------------
DERIVS = {nm for nm in ALL_NAMES if nm.startswith("D_")}
ODD_SITE, EVEN_SITE = set(), set()
for nm in ALL_NAMES:
    if nm.startswith("Y^dagger") or nm.startswith("psi^{dagger"):
        EVEN_SITE.add(nm)
    elif nm.startswith("Y^") or nm.startswith("psi_"):
        if nm not in DERIVS:
            ODD_SITE.add(nm)

def split_letters(zset):
    Z = set(zset)
    odd = sorted(Z & ODD_SITE)
    even = sorted(Z & EVEN_SITE)
    deriv = sorted(Z & DERIVS)
    other = sorted(Z - set(odd) - set(even) - set(deriv))
    return odd, even, deriv, other

def is_alternating_feasible(zset):
    odd, even, _, _ = split_letters(zset)
    return (len(odd) > 0) and (len(even) > 0)

def vacuum_ok(zset):
    mode = SETTINGS["vacuum_mode"].strip()
    Z = set(zset)
    if mode == "none":
        return True
    if mode == "Y1_Yd4":
        return {"Y^1","Y^dagger_4"}.issubset(Z)
    if mode == "Y1_psid1+":
        return {"Y^1","psi^{dagger 1+}"}.issubset(Z)
    raise ValueError(f"Unknown vacuum_mode={mode}")

# ---------------------------------------------------------------------------
# Identity formulas (linear, exact)
# ---------------------------------------------------------------------------
def derived_from_spin(H1, H2, H3, b):
    J1 = QQ(1)/2*(H1 - H2 - H3) + b/4
    J2 = QQ(1)/2*(-H1 + H2 - H3) + b/4
    J3 = QQ(1)/2*(-H1 - H2 + H3) + b/4
    J4 = QQ(1)/2*(H1 + H2 + H3) + b/4
    return (J1, J2, J3, J4)

def derived_from_J(J1, J2, J3, J4):
    H1 = QQ(1)/2*( J1 - J2 - J3 + J4)
    H2 = QQ(1)/2*(-J1 + J2 - J3 + J4)
    H3 = QQ(1)/2*(-J1 - J2 + J3 + J4)
    b  = J1 + J2 + J3 + J4
    return (H1, H2, H3, b)

def pq_from_H(H1, H2, H3):
    p1 = H2 + H3
    q  = H1 - H2
    p2 = H2 - H3
    return (p1, q, p2)

# ---------------------------------------------------------------------------
# Identity checks
# ---------------------------------------------------------------------------
def check_identities_for_letter(nm):
    d = LETTERS[nm]

    needed = ["H1c","H2c","H3c","b","J1","J2","J3","J4","p1","q","p2"]
    for k in needed:
        if k not in d:
            raise ValueError(f"Letter {nm} missing {k} for identity checks")

    H1, H2, H3, b = d["H1c"], d["H2c"], d["H3c"], d["b"]
    J1, J2, J3, J4 = d["J1"], d["J2"], d["J3"], d["J4"]
    p1, q, p2 = d["p1"], d["q"], d["p2"]

    if b != J1 + J2 + J3 + J4:
        raise RuntimeError(f"[ID FAIL] {nm}: b != J1+J2+J3+J4")

    H1c2, H2c2, H3c2, b2 = derived_from_J(J1, J2, J3, J4)
    if (H1, H2, H3, b) != (H1c2, H2c2, H3c2, b2):
        raise RuntimeError(f"[ID FAIL] {nm}: (H1c,H2c,H3c,b) != derived_from_J")

    J1b, J2b, J3b, J4b = derived_from_spin(H1, H2, H3, b)
    if (J1, J2, J3, J4) != (J1b, J2b, J3b, J4b):
        raise RuntimeError(f"[ID FAIL] {nm}: (J1..J4) != derived_from_spin")

    p1b, qb, p2b = pq_from_H(H1, H2, H3)
    if (p1, q, p2) != (p1b, qb, p2b):
        raise RuntimeError(f"[ID FAIL] {nm}: (p1,q,p2) != pq_from_H")

if SETTINGS["run_identity_checks"]:
    for nm in ALL_NAMES:
        check_identities_for_letter(nm)

# ---------------------------------------------------------------------------
# Charge access
# ---------------------------------------------------------------------------
RUN_BASIS_KEY = SETTINGS["run_basis"]
RUN_COLS = basis_cols(RUN_BASIS_KEY)
K = len(RUN_COLS)

def Cvec_table(nm, cols):
    d = LETTERS[nm]
    missing = [c for c in cols if c not in d]
    if missing:
        raise ValueError(f"Letter {nm} missing cols {missing} for basis")
    return vector(QQ, [d[c] for c in cols])

def Delta(nm):
    return LETTERS[nm]["Delta"]

def P_value(nm, nvec, cols):
    return Delta(nm) - Cvec_table(nm, cols).dot_product(nvec)

# ---------------------------------------------------------------------------
# Vector printing helpers
# ---------------------------------------------------------------------------
def vec_str(v):
    return "(" + ", ".join(str(x) for x in v) + ")"

def vec_common_den_form(v):
    vals = [QQ(x) for x in v]
    dens = [x.denominator() for x in vals]
    D = ZZ(1) if not dens else lcm(dens)
    ints = [ZZ(x * D) for x in vals]
    return D, ints

def vec_str_common_den(v):
    D, ints = vec_common_den_form(v)
    if D == 1:
        return "(" + ", ".join(str(m) for m in ints) + ")"
    return f"(1/{D})*(" + ", ".join(str(m) for m in ints) + ")"

def format_vector_lines(v, prefix):
    mode = SETTINGS["n_vector_print_mode"]
    if mode not in ("fraction", "common_den", "both"):
        raise ValueError('n_vector_print_mode must be "fraction", "common_den", or "both"')

    lines = []
    if mode == "fraction":
        lines.append(f"{prefix}{vec_str(v)}")
    elif mode == "common_den":
        lines.append(f"{prefix}{vec_str_common_den(v)}")
    else:
        lines.append(f"{prefix}{vec_str(v)}")
        lines.append(" " * len(prefix) + f"{vec_str_common_den(v)}")
    return lines

def status(P):
    return "OK" if P > 0 else ("SAT" if P == 0 else "BAD")

# ---------------------------------------------------------------------------
# Generic sum-objective helper
# ---------------------------------------------------------------------------
def weighted_sum_for_coords(vals, cols, coords):
    """
    Sum selected coordinates.
    If coords is empty, sum all coordinates.
    """
    if not coords:
        return sum(vals)
    idx = [cols.index(c) for c in coords]
    return sum(vals[i] for i in idx)

def exact_sign_zero_penalties(vals, cols, prefer_exact, prefer_nonneg, prefer_nonpos, prefer_zero,
                              penalty_exact, penalty_sign, penalty_zero):
    exact_pen = 0
    sign_pen = 0
    zero_pen = 0

    for name, x in zip(cols, vals):
        if name in prefer_exact and x != QQ(prefer_exact[name]):
            exact_pen += penalty_exact
        if name in prefer_nonneg and x < 0:
            sign_pen += penalty_sign
        if name in prefer_nonpos and x > 0:
            sign_pen += penalty_sign
        if name in prefer_zero and x != 0:
            zero_pen += penalty_zero

    return exact_pen, sign_pen, zero_pen

# ---------------------------------------------------------------------------
# Evaluate mode
# ---------------------------------------------------------------------------
def print_diagnostics(nvec, cols, order=None, show_cols=True, extra_cols=None):
    if extra_cols is None:
        extra_cols = []
    extra_cols = [c for c in extra_cols if c in LETTERS[ALL_NAMES[0]]]

    header = ["Letter","Delta"]
    if show_cols:
        header += cols
    header += ["n·C","P=Delta-n·C","Status"]
    if extra_cols:
        header += extra_cols

    rows_out = []
    if order is None:
        order = ALL_NAMES
    for nm in order:
        d = LETTERS[nm]
        ndot = Cvec_table(nm, cols).dot_product(nvec)
        P = Delta(nm) - ndot
        row = [nm, str(Delta(nm))]
        if show_cols:
            row += [str(d[c]) for c in cols]
        row += [str(ndot), str(P), status(P)]
        if extra_cols:
            row += [str(d[c]) for c in extra_cols]
        rows_out.append(row)

    widths = [max(len(h), max(len(r[i]) for r in rows_out)) for i,h in enumerate(header)]
    print(" | ".join(header[i].ljust(widths[i]) for i in range(len(header))))
    print("-" * (sum(widths) + 3*(len(widths)-1)))
    for r in rows_out:
        print(" | ".join(r[i].ljust(widths[i]) for i in range(len(r))))

    for line in format_vector_lines(vector(QQ, nvec), f"n_{RUN_BASIS_KEY} = "):
        print(line)

if SETTINGS["evaluate_mode"]:
    if SETTINGS["evaluate_n"] is None:
        raise ValueError("evaluate_mode=True but evaluate_n=None")
    n_input = [QQ(x) for x in SETTINGS["evaluate_n"]]
    if len(n_input) != K:
        raise ValueError(f"evaluate_n length {len(n_input)} != dim({RUN_BASIS_KEY})={K}")
    nvec = vector(QQ, n_input)
    print("\n=== EVALUATE MODE ===")
    print("run_basis =", RUN_BASIS_KEY)
    print("RUN_COLS  =", RUN_COLS)
    print_diagnostics(
        nvec,
        cols=RUN_COLS,
        show_cols=SETTINGS["print_charge_cols_in_diagnostics"],
        extra_cols=SETTINGS["extra_cols_in_diagnostics"],
    )
    raise SystemExit

# ---------------------------------------------------------------------------
# Build polyhedron
# ---------------------------------------------------------------------------
IEQS = [(Delta(nm),) + tuple(-Cvec_table(nm, RUN_COLS)) for nm in ALL_NAMES]

EQNS = []
for col in SETTINGS["force_zero"]:
    if col not in RUN_COLS:
        raise ValueError(f'force_zero contains "{col}" not in RUN_COLS={RUN_COLS}')
    idx = RUN_COLS.index(col)
    a = [QQ(0)] * K
    a[idx] = QQ(1)
    EQNS.append((QQ(0),) + tuple(a))

Ppoly = Polyhedron(ieqs=IEQS, eqns=EQNS, base_ring=QQ)

print("\nPolyhedron dim =", Ppoly.dim(), f"(ambient={K})")
print("run_basis =", RUN_BASIS_KEY, "| RUN_COLS =", RUN_COLS)
print("force_zero =", SETTINGS["force_zero"])
print("vacuum_mode =", SETTINGS["vacuum_mode"], "| alternation_filter =", SETTINGS["require_alternating_feasible"])

# ---------------------------------------------------------------------------
# Enumerate faces -> sectors_raw
# ---------------------------------------------------------------------------
sectors_raw = {}
for d in range(Ppoly.dim() + 1):
    for face in Ppoly.faces(d):
        Fpoly = face.as_polyhedron()
        npt = Fpoly.representative_point()

        for nm in ALL_NAMES:
            if P_value(nm, npt, RUN_COLS) < 0:
                raise RuntimeError("Representative point violates P>=0")

        zeros = frozenset([nm for nm in ALL_NAMES if P_value(nm, npt, RUN_COLS) == 0])

        if not vacuum_ok(zeros):
            continue
        if SETTINGS["require_alternating_feasible"] and (not is_alternating_feasible(zeros)):
            continue

        if zeros not in sectors_raw:
            sectors_raw[zeros] = (Fpoly.dim(), npt, Fpoly)

print("Raw deduped sectors =", len(sectors_raw))

# ---------------------------------------------------------------------------
# Symmetry helpers
# ---------------------------------------------------------------------------
def _idxs_in_universal(cols):
    return [UNIVERSAL_ORDER.index(c) for c in cols]

def subspace_invariant(M_univ, cols):
    idx = _idxs_in_universal(cols)
    outside = [j for j in range(mU) if j not in idx]
    if not outside:
        return True
    for i in idx:
        for j in outside:
            if M_univ[i, j] != 0:
                return False
    return True

def restrict_matrix(M_univ, cols):
    idx = _idxs_in_universal(cols)
    return M_univ.matrix_from_rows_and_columns(idx, idx)

def invT_if_possible(M):
    try:
        return M.transpose().inverse()
    except Exception:
        return None

def transform_n_under_action(nvec, M_univ, cols):
    M_sub = restrict_matrix(M_univ, cols)
    return (M_sub.transpose().inverse()) * vector(QQ, list(nvec))

def compose_actions(a2, a1):
    mp1, M1 = a1
    mp2, M2 = a2
    mp = {nm: mp2[mp1[nm]] for nm in ALL_NAMES}
    return (mp, M2*M1)

def action_key(act):
    mp, M = act
    return (tuple(mp[nm] for nm in ALL_NAMES), tuple(M.list()))

def group_closure(generators):
    G = [({nm:nm for nm in ALL_NAMES}, identity_matrix(QQ, mU))]
    seen = {action_key(G[0])}
    changed = True
    while changed:
        changed = False
        for g in generators:
            for a in list(G):
                h = compose_actions(g, a)
                k = action_key(h)
                if k not in seen:
                    seen.add(k)
                    G.append(h)
                    changed = True
    return G

def _apply_name_map(zset, mp):
    return frozenset(mp.get(nm, nm) for nm in zset)

_pat_Y    = re.compile(r"^Y\^([1-4])$")
_pat_Yd   = re.compile(r"^Y\^dagger_([1-4])$")
_pat_psi  = re.compile(r"^psi_\{([1-4])([+-])\}$")
_pat_psid = re.compile(r"^psi\^\{dagger ([1-4])([+-])\}$")

def permute_index_in_name(nm, perm):
    m = _pat_Y.match(nm)
    if m: return f"Y^{perm[int(m.group(1))]}"
    m = _pat_Yd.match(nm)
    if m: return f"Y^dagger_{perm[int(m.group(1))]}"
    m = _pat_psi.match(nm)
    if m:
        k = perm[int(m.group(1))]
        sgn = m.group(2)
        return f"psi_{{{k}{sgn}}}"
    m = _pat_psid.match(nm)
    if m:
        k = perm[int(m.group(1))]
        sgn = m.group(2)
        return f"psi^{{dagger {k}{sgn}}}"
    return nm

def pm_flip_name(nm):
    if nm == "D_+": return "D_-"
    if nm == "D_-": return "D_+"
    m = _pat_psi.match(nm)
    if m:
        k, sgn = m.group(1), m.group(2)
        return f"psi_{{{k}{'-' if sgn=='+' else '+'}}}"
    m = _pat_psid.match(nm)
    if m:
        k, sgn = m.group(1), m.group(2)
        return f"psi^{{dagger {k}{'-' if sgn=='+' else '+'}}}"
    return nm

def dagger_swap_name(nm):
    m = _pat_Y.match(nm)
    if m: return f"Y^dagger_{m.group(1)}"
    m = _pat_Yd.match(nm)
    if m: return f"Y^{m.group(1)}"
    m = _pat_psi.match(nm)
    if m: return f"psi^{{dagger {m.group(1)}{m.group(2)}}}"
    m = _pat_psid.match(nm)
    if m: return f"psi_{{{m.group(1)}{m.group(2)}}}"
    return nm

def action_pm_flip():
    mp = {nm: pm_flip_name(nm) for nm in ALL_NAMES}
    M = identity_matrix(QQ, mU)
    M[UNIVERSAL_ORDER.index("s"), UNIVERSAL_ORDER.index("s")] = -1
    return (mp, M)

def action_dagger_swap():
    mp = {nm: dagger_swap_name(nm) for nm in ALL_NAMES}
    M = identity_matrix(QQ, mU)
    flip_keys = ["H1c","H2c","H3c","J1","J2","J3","J4","p1","q","p2","b"]
    for k in flip_keys:
        M[UNIVERSAL_ORDER.index(k), UNIVERSAL_ORDER.index(k)] = -1
    return (mp, M)

Wfund = {
    1: vector(QQ, [ 1,  0,  0]),
    2: vector(QQ, [-1,  1,  0]),
    3: vector(QQ, [ 0, -1,  1]),
    4: vector(QQ, [ 0,  0, -1]),
}
W_basis = matrix(QQ, 3, 3, [Wfund[1], Wfund[2], Wfund[3]]).transpose()
W_basis_inv = W_basis.inverse()

def T_from_perm(perm):
    Wp = matrix(QQ, 3, 3, [Wfund[perm[1]], Wfund[perm[2]], Wfund[perm[3]]]).transpose()
    return Wp * W_basis_inv

A_pq_from_H = matrix(QQ, [
    [0, 1, 1],
    [1,-1, 0],
    [0, 1,-1],
])
A_H_from_pq = A_pq_from_H.inverse()

def action_S4_perm(perm):
    mp = {nm: permute_index_in_name(nm, perm) for nm in ALL_NAMES}
    Tpq = T_from_perm(perm)
    TH  = A_H_from_pq * Tpq * A_pq_from_H

    M = identity_matrix(QQ, mU)

    for i, ik in enumerate(["p1","q","p2"]):
        for j, jk in enumerate(["p1","q","p2"]):
            M[UNIVERSAL_ORDER.index(ik), UNIVERSAL_ORDER.index(jk)] = Tpq[i,j]

    for i, ik in enumerate(["H1c","H2c","H3c"]):
        for j, jk in enumerate(["H1c","H2c","H3c"]):
            M[UNIVERSAL_ORDER.index(ik), UNIVERSAL_ORDER.index(jk)] = TH[i,j]

    P = matrix(QQ, 4, 4, 0)
    for old in [1,2,3,4]:
        new = perm[old]
        P[new-1, old-1] = 1
    for i in range(4):
        for j in range(4):
            M[UNIVERSAL_ORDER.index(f"J{i+1}"), UNIVERSAL_ORDER.index(f"J{j+1}")] = P[i,j]

    return (mp, M)

# ---------------------------------------------------------------------------
# Scoring functions
# ---------------------------------------------------------------------------
def preference_score(zset):
    Z = set(zset)
    has = lambda x: 1 if x in Z else 0
    return (
        -has("Y^1"),
        -has("Y^dagger_4"),
        -has("psi_{4+}"),
        -has("psi^{dagger 1+}"),
        -has("Y^2"),
        sum(has(x) for x in ["Y^dagger_1","Y^dagger_2","Y^dagger_3"]),
        sum(has(x) for x in ["psi_{2+}","psi_{3+}"]),
        sum(has(x) for x in ["psi^{dagger 2+}","psi^{dagger 3+}"]),
        len(Z),
        tuple(sorted(Z)),
    )

def generic_nvec_score(nvec, cols, *,
                       prefer_exact_values,
                       prefer_nonnegative,
                       prefer_nonpositive,
                       prefer_zero,
                       penalty_exact,
                       penalty_sign,
                       penalty_zero,
                       sum_mode,
                       sum_coords):
    """
    Generic score builder for any n-vector preference scheme.

    Lower score is better.
    """
    vals = [QQ(x) for x in nvec]

    exact_pen, sign_pen, zero_pen = exact_sign_zero_penalties(
        vals, cols,
        prefer_exact=prefer_exact_values,
        prefer_nonneg=set(prefer_nonnegative),
        prefer_nonpos=set(prefer_nonpositive),
        prefer_zero=set(prefer_zero),
        penalty_exact=penalty_exact,
        penalty_sign=penalty_sign,
        penalty_zero=penalty_zero,
    )

    neg_count = sum(1 for x in vals if x < 0)
    pos_count = sum(1 for x in vals if x > 0)
    zero_count = sum(1 for x in vals if x == 0)

    if sum_mode == "none":
        sum_term = QQ(0)
    elif sum_mode == "maximize":
        sum_term = -weighted_sum_for_coords(vals, cols, sum_coords)
    elif sum_mode == "minimize":
        sum_term = weighted_sum_for_coords(vals, cols, sum_coords)
    else:
        raise ValueError('sum_mode must be "none", "maximize", or "minimize"')

    return (
        exact_pen,
        sign_pen,
        zero_pen,
        neg_count,         # fewer negatives
        -pos_count,        # more positives
        -zero_count,       # more zeros
        sum_term,          # chosen sum objective
        tuple(vals),       # deterministic tie-break
    )

def canonical_nvec_score(nvec, cols):
    return generic_nvec_score(
        nvec, cols,
        prefer_exact_values=SETTINGS["nvec_prefer_exact_values"],
        prefer_nonnegative=SETTINGS["nvec_prefer_nonnegative"],
        prefer_nonpositive=SETTINGS["nvec_prefer_nonpositive"],
        prefer_zero=SETTINGS["nvec_prefer_zero"],
        penalty_exact=SETTINGS["nvec_penalty_exact"],
        penalty_sign=SETTINGS["nvec_penalty_sign"],
        penalty_zero=SETTINGS["nvec_penalty_zero"],
        sum_mode=SETTINGS["nvec_sum_mode"],
        sum_coords=SETTINGS["nvec_sum_coords"],
    )

# ---------------------------------------------------------------------------
# Nice / face-point search helpers
# ---------------------------------------------------------------------------
def relative_interior_contains(Fpoly, x):
    if Fpoly.dim() == 0:
        for h in Fpoly.equations():
            if h.A().dot_product(x) + h.b() != 0:
                return False
        for h in Fpoly.inequalities():
            if h.A().dot_product(x) + h.b() < 0:
                return False
        return True

    for h in Fpoly.equations():
        if h.A().dot_product(x) + h.b() != 0:
            return False
    for h in Fpoly.inequalities():
        if h.A().dot_product(x) + h.b() <= 0:
            return False
    return True

def common_denominator_key(v):
    vals = [QQ(x) for x in v if x != 0]
    if not vals:
        return ZZ(1), ZZ(0), ZZ(0)
    dens = [x.denominator() for x in vals]
    return lcm(dens), sum(dens), len(set(dens))

def nice_point_score(v, varnames):
    vals = [QQ(x) for x in v]
    nonzero = [x for x in vals if x != 0]

    base = generic_nvec_score(
        vals, varnames,
        prefer_exact_values=SETTINGS["nice_prefer_exact_values"],
        prefer_nonnegative=SETTINGS["nice_prefer_nonnegative"],
        prefer_nonpositive=SETTINGS["nice_prefer_nonpositive"],
        prefer_zero=SETTINGS["nice_prefer_zero"],
        penalty_exact=SETTINGS["nice_penalty_exact"],
        penalty_sign=SETTINGS["nice_penalty_sign"],
        penalty_zero=SETTINGS["nice_penalty_zero"],
        sum_mode=SETTINGS["nice_sum_mode"],
        sum_coords=SETTINGS["nice_sum_coords"],
    )

    distinct_abs_count = len(set(abs(x) for x in nonzero))
    den_lcm, den_sum, den_distinct = common_denominator_key(vals)
    total_abs = sum(abs(x) for x in nonzero)
    num_sum = sum(abs(x.numerator()) for x in nonzero)

    # Put the generic soft preferences first, then pretty-specific tie-breaks.
    return (
        *base[:-1],             # penalties + sign/zero/sum terms
        distinct_abs_count,     # equal up to sign preferred
        den_distinct,           # same denominator preferred
        den_lcm,                # smaller common denominator preferred
        den_sum,
        -total_abs,
        num_sum,
        base[-1],               # deterministic tuple(vals)
    )

def preferred_integer_values():
    M = SETTINGS["nice_max_integer"]
    vals = [0]
    vals += list(range(1, M + 1))
    vals += list(range(-1, -M - 1, -1))
    return vals

def integer_patterns_for_support_size(r):
    nonzero_vals = [x for x in preferred_integer_values() if x != 0]
    for tup in product(nonzero_vals, repeat=r):
        yield tup

def candidate_vectors_for_face_common_den(Kdim):
    D_list = SETTINGS["nice_common_denominators"]

    for D in D_list:
        for r in range(0, Kdim + 1):
            for support in combinations(range(Kdim), r):
                for coeffs in integer_patterns_for_support_size(r):
                    m = [0] * Kdim
                    for idx, c in zip(support, coeffs):
                        m[idx] = c
                    yield vector(QQ, [QQ(mi) / QQ(D) for mi in m])

def search_best_face_point(Fpoly, fallback_point, score_fn, enabled, label):
    if not enabled:
        return (fallback_point, f"raw ({label}-search off)")

    if Fpoly.dim() == 0:
        return (fallback_point, f"raw (0-dim face)")

    best = None
    best_score = None
    tested = 0
    max_test = SETTINGS["nice_max_candidates_per_face"]

    for cand in candidate_vectors_for_face_common_den(K):
        tested += 1
        if max_test is not None and tested > max_test:
            break

        if relative_interior_contains(Fpoly, cand):
            sc = score_fn(cand)
            if best is None or sc < best_score:
                best = cand
                best_score = sc

    if best is not None:
        return (best, f"{label}-search ({tested} candidates tested)")
    return (fallback_point, f"raw ({label}-search failed after {tested} candidates)")

# ---------------------------------------------------------------------------
# Face cache
# ---------------------------------------------------------------------------
FACE_CACHE = {}

def build_face_polyhedron_from_zero_set(zset):
    eqs_z = [(Delta(nm),) + tuple(-Cvec_table(nm, RUN_COLS)) for nm in zset]
    return Polyhedron(ieqs=IEQS, eqns=EQNS + eqs_z, base_ring=QQ)

def get_face_record(zset):
    if zset in FACE_CACHE:
        return FACE_CACHE[zset]

    if zset in sectors_raw:
        fdim, raw_npt, Fpoly = sectors_raw[zset]
    else:
        Fpoly = build_face_polyhedron_from_zero_set(zset)
        raw_npt = Fpoly.representative_point()
        fdim = Fpoly.dim()

    pretty_npt, pretty_note = search_best_face_point(
        Fpoly,
        raw_npt,
        score_fn=lambda v: nice_point_score(v, RUN_COLS),
        enabled=SETTINGS["use_nice_example_interiors"],
        label="pretty"
    )

    need_nvec_point = (
        SETTINGS["canonical_rep_mode"] == "nvec"
        or SETTINGS["final_sector_sort_mode"] == "nvec"
        or SETTINGS["example_point_mode"] == "nvec"
    )

    nvec_npt, nvec_note = search_best_face_point(
        Fpoly,
        raw_npt,
        score_fn=lambda v: canonical_nvec_score(v, RUN_COLS),
        enabled=need_nvec_point,
        label="nvec"
    )

    rec = {
        "zset": zset,
        "fdim": fdim,
        "Fpoly": Fpoly,
        "raw_npt": raw_npt,
        "raw_note": "raw",
        "pretty_npt": pretty_npt,
        "pretty_note": pretty_note,
        "nvec_npt": nvec_npt,
        "nvec_note": nvec_note,
    }
    FACE_CACHE[zset] = rec
    return rec

# ---------------------------------------------------------------------------
# Quotienting
# ---------------------------------------------------------------------------
any_quotients = (
    SETTINGS["quotient_pm_flip"]
    or SETTINGS["quotient_dagger_swap"]
    or SETTINGS["quotient_index_perm_S4"]
)

if not any_quotients:
    canonical_zsets = set(sectors_raw.keys())
else:
    gens = []
    if SETTINGS["quotient_index_perm_S4"]:
        for p in permutations([1,2,3,4]):
            perm = {1:p[0],2:p[1],3:p[2],4:p[3]}
            gens.append(action_S4_perm(perm))
    if SETTINGS["quotient_pm_flip"]:
        gens.append(action_pm_flip())
    if SETTINGS["quotient_dagger_swap"]:
        gens.append(action_dagger_swap())

    ALL_ACTIONS = group_closure(gens)

    valid_actions = []
    for (mp, M_univ) in ALL_ACTIONS:
        if not subspace_invariant(M_univ, RUN_COLS):
            continue
        if invT_if_possible(restrict_matrix(M_univ, RUN_COLS)) is None:
            continue
        valid_actions.append((mp, M_univ))

    if not valid_actions:
        raise RuntimeError("No valid symmetry actions preserve this run basis; cannot quotient.")

    canonical_zsets = set()

    for zset, (fdim, npt, Fpoly) in sectors_raw.items():
        best = None
        best_key = None

        for (mp, M_univ) in valid_actions:
            z_img = _apply_name_map(zset, mp)
            mode = SETTINGS["canonical_rep_mode"]

            if mode == "letters":
                key = (
                    preference_score(z_img)
                    if SETTINGS["use_preferences_for_canonical_rep"]
                    else (tuple(sorted(z_img)),)
                )

            elif mode == "nvec":
                rec = get_face_record(z_img)
                key = canonical_nvec_score(rec["nvec_npt"], RUN_COLS) + (tuple(sorted(z_img)),)

            else:
                raise ValueError(f"Unknown canonical_rep_mode={mode}")

            if best is None or key < best_key:
                best, best_key = z_img, key

        canonical_zsets.add(best)

    print("After quotienting, sectors_canonical =", len(canonical_zsets))

# ---------------------------------------------------------------------------
# Final sectors = cached face records for canonical zsets
# ---------------------------------------------------------------------------
sectors_final = {zset: get_face_record(zset) for zset in canonical_zsets}

# ---------------------------------------------------------------------------
# Conversion between bases
# ---------------------------------------------------------------------------
_CONV_CACHE = {}

def charge_matrix_from_table(cols):
    k = len(cols)
    N = len(ALL_NAMES)
    A = matrix(QQ, k, N)
    for j, nm in enumerate(ALL_NAMES):
        A.set_column(j, Cvec_table(nm, cols))
    return A

def master_cols(master):
    if master == "spin":
        return ["H1c","H2c","H3c","b","s"]
    elif master == "J":
        return ["J1","J2","J3","J4","s"]
    else:
        raise ValueError("master_basis must be 'spin' or 'J'")

def basis_to_master_matrix(bkey, master):
    cols_basis = basis_cols(bkey)
    cols_master = master_cols(master)
    km = len(cols_master)
    kb = len(cols_basis)

    idx = {c:i for i,c in enumerate(cols_master)}
    M = matrix(QQ, kb, km, 0)

    for r_i, c in enumerate(cols_basis):
        if c in cols_master:
            M[r_i, idx[c]] = 1
            continue

        if master == "spin":
            H1 = "H1c"; H2 = "H2c"; H3 = "H3c"; B = "b"

            if c == "p1":
                M[r_i, idx[H2]] = 1
                M[r_i, idx[H3]] = 1

            elif c == "q":
                M[r_i, idx[H1]] = 1
                M[r_i, idx[H2]] = -1

            elif c == "p2":
                M[r_i, idx[H2]] = 1
                M[r_i, idx[H3]] = -1

            elif c in ["J1","J2","J3","J4"]:
                if c == "J1":
                    M[r_i, idx[H1]] = QQ(1)/2
                    M[r_i, idx[H2]] = -QQ(1)/2
                    M[r_i, idx[H3]] = -QQ(1)/2
                    M[r_i, idx[B]]  = QQ(1)/4
                if c == "J2":
                    M[r_i, idx[H1]] = -QQ(1)/2
                    M[r_i, idx[H2]] = QQ(1)/2
                    M[r_i, idx[H3]] = -QQ(1)/2
                    M[r_i, idx[B]]  = QQ(1)/4
                if c == "J3":
                    M[r_i, idx[H1]] = -QQ(1)/2
                    M[r_i, idx[H2]] = -QQ(1)/2
                    M[r_i, idx[H3]] = QQ(1)/2
                    M[r_i, idx[B]]  = QQ(1)/4
                if c == "J4":
                    M[r_i, idx[H1]] = QQ(1)/2
                    M[r_i, idx[H2]] = QQ(1)/2
                    M[r_i, idx[H3]] = QQ(1)/2
                    M[r_i, idx[B]]  = QQ(1)/4
            else:
                raise ValueError(f"Don't know how to derive {c} from spin master")

        else:
            J1="J1"; J2="J2"; J3="J3"; J4="J4"

            if c == "b":
                M[r_i, idx[J1]] = 1
                M[r_i, idx[J2]] = 1
                M[r_i, idx[J3]] = 1
                M[r_i, idx[J4]] = 1

            elif c in ["H1c","H2c","H3c"]:
                if c == "H1c":
                    M[r_i, idx[J1]] = QQ(1)/2
                    M[r_i, idx[J2]] = -QQ(1)/2
                    M[r_i, idx[J3]] = -QQ(1)/2
                    M[r_i, idx[J4]] = QQ(1)/2
                if c == "H2c":
                    M[r_i, idx[J1]] = -QQ(1)/2
                    M[r_i, idx[J2]] = QQ(1)/2
                    M[r_i, idx[J3]] = -QQ(1)/2
                    M[r_i, idx[J4]] = QQ(1)/2
                if c == "H3c":
                    M[r_i, idx[J1]] = -QQ(1)/2
                    M[r_i, idx[J2]] = -QQ(1)/2
                    M[r_i, idx[J3]] = QQ(1)/2
                    M[r_i, idx[J4]] = QQ(1)/2

            elif c in ["p1","q","p2"]:
                if c == "p1":
                    M[r_i, idx[J1]] = -1
                    M[r_i, idx[J4]] = 1
                if c == "q":
                    M[r_i, idx[J1]] = 1
                    M[r_i, idx[J2]] = -1
                if c == "p2":
                    M[r_i, idx[J2]] = 1
                    M[r_i, idx[J3]] = -1
            else:
                raise ValueError(f"Don't know how to derive {c} from J master")

    return M

def convert_n_invertible(nvec, from_basis, to_basis):
    cols_from = basis_cols(from_basis)
    cols_to   = basis_cols(to_basis)
    if len(cols_from) != len(cols_to):
        return (None, "n/a (dimension mismatch)")

    if SETTINGS["identities_mode"] == "use_for_conversion":
        master = SETTINGS["master_basis"]
        M_from = basis_to_master_matrix(from_basis, master)
        M_to   = basis_to_master_matrix(to_basis, master)

        if M_from.nrows() != M_from.ncols():
            return (None, "n/a (from-basis not square in master)")
        if M_from.det() == 0:
            return (None, "n/a (from-basis not invertible in master)")

        L = M_to * M_from.inverse()

        if L.det() == 0:
            return (None, "n/a (map singular)")
        n_to = (L.transpose().inverse()) * vector(QQ, list(nvec))
        return (n_to, "")

    key = (from_basis, to_basis)
    if key in _CONV_CACHE:
        L = _CONV_CACHE[key]
    else:
        A = charge_matrix_from_table(cols_from)
        B = charge_matrix_from_table(cols_to)
        kdim = A.nrows()
        AT = A.transpose()
        if AT.rank() < kdim:
            _CONV_CACHE[key] = None
            L = None
        else:
            try:
                Y = AT.solve_left(B.transpose())
                L = Y.transpose()
                if L.det() == 0 or L*A != B:
                    L = None
            except Exception:
                L = None
            _CONV_CACHE[key] = L

    if L is None:
        return (None, "n/a (not invertible)")
    n_to = (L.transpose().inverse()) * vector(QQ, list(nvec))
    return (n_to, "")

def print_n_in_other_bases(npt):
    print("  n in other bases (invertible-only):")
    for bkey in sorted(BASES.keys()):
        if bkey == RUN_BASIS_KEY:
            continue
        v, note = convert_n_invertible(vector(QQ, list(npt)), RUN_BASIS_KEY, bkey)
        if v is None:
            print(f"    {bkey:6s}: (n/a) [{note}]")
        else:
            lines = format_vector_lines(v, f"    {bkey:6s}: ")
            for line in lines:
                print(line)

# ---------------------------------------------------------------------------
# Printing sectors
# ---------------------------------------------------------------------------
def pretty_linexpr(a, b0, varnames):
    pieces = []
    for ci, var in zip(a, varnames):
        if ci == 0:
            continue
        abs_ci = abs(ci)
        term = var if abs_ci == 1 else f"{abs_ci}*{var}"
        if not pieces:
            pieces.append(term if ci > 0 else f"-{term}")
        else:
            pieces.append(f"+ {term}" if ci > 0 else f"- {term}")
    if b0 != 0:
        const = str(abs(b0))
        if not pieces:
            pieces.append(str(b0))
        else:
            pieces.append(f"+ {const}" if b0 > 0 else f"- {const}")
    return " ".join(pieces) if pieces else "0"

def choose_example_point_record(rec):
    mode = SETTINGS["example_point_mode"]
    if mode == "pretty":
        return rec["pretty_npt"], rec["pretty_note"]
    if mode == "nvec":
        return rec["nvec_npt"], rec["nvec_note"]
    if mode == "raw":
        return rec["raw_npt"], rec["raw_note"]
    raise ValueError(f"Unknown example_point_mode={mode}")

def describe_face(Fpoly, example_npt, max_ineq=12):
    out = []
    out.append(f"  Face dim = {Fpoly.dim()}")

    v = vector(QQ, list(example_npt))
    out.extend(format_vector_lines(v, f"  Example interior n_{RUN_BASIS_KEY} = "))

    ineqs = list(Fpoly.inequalities())
    if ineqs:
        out.append("  Interior constraints (strict):")
        scored = []
        for h in ineqs:
            val = h.A().dot_product(example_npt) + h.b()
            scored.append((val, h))
        scored.sort(key=lambda t: t[0])
        for _, h in scored[:max_ineq]:
            out.append("    " + pretty_linexpr(list(h.A()), h.b(), RUN_COLS) + " > 0")
        if len(ineqs) > max_ineq:
            out.append(f"    ... ({len(ineqs)-max_ineq} more)")
    return "\n".join(out)

# ---------------------------------------------------------------------------
# Final sector ordering
# ---------------------------------------------------------------------------
def final_sector_sort_key(item):
    zset, rec = item
    mode = SETTINGS["final_sector_sort_mode"]

    if mode == "nvec":
        return canonical_nvec_score(rec["nvec_npt"], RUN_COLS) + (tuple(sorted(zset)),)

    return (tuple(sorted(zset)),)

items = sorted(sectors_final.items(), key=final_sector_sort_key)
if SETTINGS["max_sectors_to_print"] is not None:
    items = items[:SETTINGS["max_sectors_to_print"]]

print("\n=== SECTORS ===\n")
for i, (zset, rec) in enumerate(items, 1):
    fdim = rec["fdim"]
    Fpoly = rec["Fpoly"]
    raw_npt = rec["raw_npt"]

    example_npt, example_note = choose_example_point_record(rec)

    odd, even, deriv, other = split_letters(zset)
    print(f"Sector {i} (face dim {fdim}):")
    print("  Odd letters :", odd)
    print("  Even letters:", even)
    print("  Derivatives :", deriv)
    if other:
        print("  Other       :", other)

    print(describe_face(Fpoly, example_npt, max_ineq=SETTINGS["max_strict_ineq_to_print"]))
    print("  Example-point source:", example_note)

    if SETTINGS["print_raw_representative_if_different"] and raw_npt != example_npt:
        raw_v = vector(QQ, list(raw_npt))
        for line in format_vector_lines(raw_v, "  Raw Sage representative = "):
            print(line)

    if SETTINGS["print_n_in_other_bases"]:
        print_n_in_other_bases(example_npt)
    print()

print("Summary:")
print("  raw sectors:", len(sectors_raw))
print("  final sectors:", len(sectors_final))


Polyhedron dim = 5 (ambient=5)
run_basis = J5 | RUN_COLS = ['J1', 'J2', 'J3', 'J4', 's']
force_zero = []
vacuum_mode = none | alternation_filter = True
Raw deduped sectors = 214
After quotienting, sectors_canonical = 10

=== SECTORS ===

Sector 1 (face dim 0):
  Odd letters : ['Y^1', 'Y^2', 'psi_{3+}', 'psi_{4+}']
  Even letters: ['Y^dagger_3', 'Y^dagger_4', 'psi^{dagger 1+}', 'psi^{dagger 2+}']
  Derivatives : ['D_+']
  Face dim = 0
  Example interior n_J5 = (1/2, -1/2, -1/2, 1/2, 1)
  Example-point source: raw (pretty-search off)

Sector 2 (face dim 1):
  Odd letters : ['Y^1', 'psi_{4+}']
  Even letters: ['Y^dagger_4', 'psi^{dagger 1+}']
  Derivatives : ['D_+']
  Face dim = 1
  Example interior n_J5 = (0, 0, -1/2, 1/2, 1)
  Interior constraints (strict):
    2*J1 + 1 > 0
    -2*J1 + 1 > 0
  Example-point source: raw (pretty-search off)

Sector 3 (face dim 2):
  Odd letters : ['Y^1']
  Even letters: ['psi^{dagger 1+}']
  Derivatives : ['D_+']
  Face dim = 2
  Example interior n_J5 = 